# Bài 60: Mô hình sinh và Mô hình khuếch tán (Generative and Diffusion Models)

Mọi mạng từ đầu đến giờ đều ánh xạ một ảnh thành thứ gì đó nhỏ hơn: một nhãn, một hộp, một mặt nạ, một vector nhúng. **Mô hình khuếch tán (diffusion models)** (<a href="../references.html#sohl-dickstein-2015">Sohl-Dickstein et al., 2015</a>; <a href="../references.html#ho-2020-ddpm">Ho et al., 2020</a>) chạy ý tưởng theo chiều ngược lại: học cách ánh xạ nhiễu thuần túy thành một ảnh chân thực. Mẹo cốt lõi đơn giản một cách đánh lừa — huấn luyện một mạng hoàn tác mỗi lần một bước nhỏ của sự làm hỏng bằng nhiễu, rồi nối chuỗi nhiều bước như vậy, bắt đầu từ nhiễu và kết thúc ở thứ gì đó trông giống dữ liệu huấn luyện.

In [ ]:
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Quá trình thuận: phá hủy dần một ảnh

Định nghĩa một lịch trình cố định gồm $T$ bước, mỗi bước thêm một lượng nhỏ nhiễu Gauss: $x_t = \sqrt{\alpha_t}\,x_{t-1} + \sqrt{1-\alpha_t}\,\varepsilon$. Áp dụng điều này lặp đi lặp lại thì mô phỏng từng bước một rất chậm, nhưng vì tổng của các biến Gauss độc lập tự nó là Gauss, có một **dạng đóng (closed form)** nhảy thẳng từ ảnh gốc $x_0$ tới bất kỳ bước $t$ nào: $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$, trong đó $\bar\alpha_t = \prod_{s=1}^t \alpha_s$.

In [ ]:
SIZE = 16
T = 100

betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

def make_image(shape_type, cx, cy, size=SIZE):
    img = np.zeros((size, size), dtype=np.float32)
    if shape_type == 'plus':
        img[cy-1:cy+2, cx-3:cx+4] = 1.0
        img[cy-3:cy+4, cx-1:cx+2] = 1.0
    else:
        yy, xx = np.mgrid[0:size, 0:size]
        img[((xx-cx)**2 + (yy-cy)**2) <= 9] = 1.0
    return img * 2 - 1  # scale to [-1, 1], the usual diffusion-model convention

def make_dataset(rng, n):
    imgs = []
    for _ in range(n):
        shape_type = rng.choice(['plus', 'circle'])
        cx, cy = rng.integers(5, 11), rng.integers(5, 11)
        imgs.append(make_image(shape_type, cx, cy))
    return np.array(imgs, dtype=np.float32)

rng = np.random.default_rng(2)
X = make_dataset(rng, 500)
X_t = torch.tensor(X).unsqueeze(1)

# validate the closed form against literally simulating T small steps, statistically
x0_single = X_t[0:1]
t_idx = 50
n_trials = 3000
x0_rep = x0_single.expand(n_trials, -1, -1, -1)

torch.manual_seed(1)
x_t_iterative = x0_rep.clone()
for step in range(t_idx):
    x_t_iterative = torch.sqrt(alphas[step]) * x_t_iterative + torch.sqrt(betas[step]) * torch.randn_like(x0_rep)

torch.manual_seed(2)
eps = torch.randn_like(x0_rep)
x_t_closed = torch.sqrt(alpha_bars[t_idx - 1]) * x0_rep + torch.sqrt(1 - alpha_bars[t_idx - 1]) * eps

print(f'{t_idx} iterative small steps:  mean={x_t_iterative.mean().item():.4f}, std={x_t_iterative.std().item():.4f}')
print(f'closed-form single jump:      mean={x_t_closed.mean().item():.4f}, std={x_t_closed.std().item():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(11, 2))
x0_demo = X_t[0:1]
for ax, t_show in zip(axes, [0, 10, 25, 50, 75, 99]):
    torch.manual_seed(0)
    if t_show == 0:
        img = x0_demo
    else:
        img = torch.sqrt(alpha_bars[t_show]) * x0_demo + torch.sqrt(1 - alpha_bars[t_show]) * torch.randn_like(x0_demo)
    ax.imshow(img[0, 0], cmap='gray'); ax.set_title(f't={t_show}', fontsize=8); ax.axis('off')
plt.suptitle('Forward process: the same image at increasing noise levels')
plt.show()

## Huấn luyện: dự đoán nhiễu, không phải ảnh

Nhiệm vụ của mạng phát biểu rất đơn giản: cho một ảnh nhiễu $x_t$ và mức nhiễu $t$, dự đoán nhiễu $\varepsilon$ đã được thêm vào. Tín hiệu huấn luyện có sẵn miễn phí — chọn một ảnh ngẫu nhiên và một $t$ ngẫu nhiên, thêm nhiễu đã biết qua dạng đóng ở trên, và kiểm tra xem mạng có khôi phục được nó không. Hoàn toàn không cần nhãn, chỉ cần ảnh không nhãn (cùng tinh thần tự giám sát như học tương phản ở Bài 49, áp dụng cho một nhiệm vụ hoàn toàn khác). Mạng cần biết $t$ vì *lượng* hiệu chỉnh đúng cần áp dụng rất khác nhau giữa một ảnh chỉ hơi nhiễu và một ảnh gần như toàn nhiễu — điều này được đưa vào qua một vector nhúng thời gian hình sin, cùng cách xây dựng như mã hóa vị trí ở Bài 46, đánh chỉ số "đã đi được bao xa" thay vì "ở đâu trong chuỗi".

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim=32):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-np.log(10000) * torch.arange(half).float() / half)
        args = t[:, None].float() * freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class DenoiseNet(nn.Module):
    def __init__(self, ch=32, time_dim=32):
        super().__init__()
        self.time_embed = TimeEmbedding(time_dim)
        self.time_mlp = nn.Linear(time_dim, ch)
        self.conv1 = nn.Conv2d(1, ch, 3, padding=1)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1)
        self.conv3 = nn.Conv2d(ch, 1, 3, padding=1)

    def forward(self, x, t):
        temb = self.time_mlp(self.time_embed(t))[:, :, None, None]
        h = F.relu(self.conv1(x) + temb)
        h = F.relu(self.conv2(h) + temb)
        return self.conv3(h)

torch.manual_seed(0)
model = DenoiseNet()
opt = torch.optim.Adam(model.parameters(), lr=0.001)
n = len(X_t)
for epoch in range(800):
    idx = np.random.default_rng(epoch).permutation(n)[:64]
    x0 = X_t[idx]
    t = torch.randint(0, T, (x0.shape[0],))
    noise = torch.randn_like(x0)
    ab = alpha_bars[t][:, None, None, None]
    x_t_batch = torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * noise
    pred_noise = model(x_t_batch, t)
    loss = F.mse_loss(pred_noise, noise)
    opt.zero_grad()
    loss.backward()
    opt.step()

print(f'final training loss (predicted-vs-true noise MSE): {loss.item():.3f}')

## Lấy mẫu: đảo ngược quá trình, mỗi lần loại bỏ một nhiễu dự đoán

Bắt đầu từ nhiễu Gauss thuần túy, liên tục hỏi mạng đã huấn luyện "nhiễu nào đã được thêm vào để đến được đây?", trừ đi một phiên bản đã co giãn của dự đoán đó, và thêm lại một lượng nhỏ ngẫu nhiên mới (trừ ở bước cuối cùng) — phần cuối này quan trọng: không có nó, quá trình là tất định và có xu hướng tạo ra các trung bình mờ thay vì các mẫu sắc nét, đa dạng.

In [ ]:
@torch.no_grad()
def sample(model, n_samples, seed):
    torch.manual_seed(seed)
    x = torch.randn(n_samples, 1, SIZE, SIZE)
    for t in reversed(range(T)):
        t_batch = torch.full((n_samples,), t, dtype=torch.long)
        pred_noise = model(x, t_batch)
        alpha_t, alpha_bar_t, beta_t = alphas[t], alpha_bars[t], betas[t]
        mean = (1 / torch.sqrt(alpha_t)) * (x - (beta_t / torch.sqrt(1 - alpha_bar_t)) * pred_noise)
        x = mean + torch.sqrt(beta_t) * torch.randn_like(x) if t > 0 else mean
    return x

gen_samples = sample(model, 8, seed=42)

fig, axes = plt.subplots(1, 8, figsize=(11, 2))
for ax, im in zip(axes, gen_samples):
    ax.imshow(im[0], cmap='gray'); ax.axis('off')
plt.suptitle('Generated samples, starting from pure noise')
plt.show()

## Các mẫu có thực sự giống các hình huấn luyện không?

Theo cách xây dựng, mọi ảnh huấn luyện thật đều là chính xác **một** đốm liên thông đặc (một dấu cộng hoặc một hình tròn). Nhiễu ngẫu nhiên thuần túy, phân ngưỡng theo cùng cách, là một tập rải rác nhiều mảnh nhỏ không liên thông. Hãy kiểm tra xem các mẫu được sinh ra nằm ở đâu trên phổ đó.

In [ ]:
def connectivity_stats(imgs):
    n_components_list = []
    for im in imgs:
        binary = (im[0].numpy() > 0).astype(np.uint8)
        n_comp, _ = cv2.connectedComponents(binary)
        n_components_list.append(n_comp - 1)  # subtract the background label
    return np.array(n_components_list)

n_eval = 50
gen_eval = sample(model, n_eval, seed=7)
noise_eval = torch.randn(n_eval, 1, SIZE, SIZE)

gen_ncomp = connectivity_stats(gen_eval)
noise_ncomp = connectivity_stats(noise_eval)
real_ncomp = connectivity_stats(X_t[:n_eval])

print(f'mean # connected foreground blobs, real training images: {real_ncomp.mean():.1f}  (always exactly 1, by construction)')
print(f'mean # connected foreground blobs, generated samples:    {gen_ncomp.mean():.1f}')
print(f'mean # connected foreground blobs, pure random noise:    {noise_ncomp.mean():.1f}')

Các mẫu được sinh ra gần với cấu trúc một đốm của ảnh thật hơn nhiều so với nhiễu thuần túy — quá trình ngược đã học cách ráp lại các hình dạng mạch lạc, liên thông từ hư không, không chỉ làm trơn cục bộ các điểm ảnh ngẫu nhiên.

## Từ mô hình đồ chơi này đến Stable Diffusion

Ba thay đổi phân biệt mô hình của bài này với một hệ thống văn-bản-thành-ảnh thực sự như Stable Diffusion:

- **Điều kiện hóa (conditioning)**: đưa một vector nhúng văn bản (bộ mã hóa văn bản CLIP ở Bài 52, hoặc một mô hình ngôn ngữ chuyên dụng) vào mạng khử nhiễu ở mỗi bước — ví dụ qua chú ý chéo (Bài 45), cùng cơ chế truy vấn/khóa/giá trị được dùng xuyên suốt Phần 4, chỉ là với *truy vấn* đến từ các đặc trưng ảnh và *khóa/giá trị* đến từ vector nhúng văn bản. Đây là điều biến "sinh một ảnh hợp lý" thành "sinh một ảnh khớp với lời nhắc này".
- **Dẫn hướng không cần bộ phân loại (classifier-free guidance)**: huấn luyện mạng cả có lẫn không có điều kiện văn bản (ngẫu nhiên bỏ nó đi trong khi huấn luyện), rồi lúc lấy mẫu ngoại suy *ra xa* dự đoán không điều kiện về phía dự đoán có điều kiện — chỉ riêng mẹo này đã chịu trách nhiệm cho một phần lớn bước nhảy vọt về chất lượng tuân thủ lời nhắc trong các mô hình văn-bản-thành-ảnh hiện đại.
- **Khuếch tán tiềm ẩn (latent diffusion)**: chạy toàn bộ quá trình khuếch tán trong một không gian tiềm ẩn nén (nút thắt cổ chai của một bộ tự mã hóa đã tiền huấn luyện, Bài 48 — không phải điểm ảnh thô) thay vì trên các ảnh độ phân giải đầy đủ — lý do Stable Diffusion có thể sinh một ảnh 512x512 với số bước hợp lý trên một GPU duy nhất, thay vì phép khuếch tán trong không gian điểm ảnh mà bài này dùng.

Quá trình thuận, mục tiêu huấn luyện dự đoán nhiễu, và vòng lặp lấy mẫu ngược lặp lại — ba thành phần được xây dựng từ đầu ở trên — không bị thay đổi bởi bất kỳ bổ sung nào trong số này. Chúng vẫn là cơ chế nằm bên dưới.

### Bài tập

1. Giảm `T` từ 100 xuống 20. Thước đo đốm liên thông cho các mẫu được sinh ra tốt hơn, tệ hơn, hay giữ nguyên — và điều đó gợi ý gì về sự đánh đổi giữa số bước khuếch tán và chất lượng mẫu?
2. Vòng lặp lấy mẫu thêm nhiễu mới ở mọi bước trừ bước cuối (`torch.sqrt(beta_t) * torch.randn_like(x) if t > 0 else mean`). Hãy loại bỏ hoàn toàn việc thêm nhiễu đó (luôn dùng `mean`) và so sánh thước đo đốm liên thông. Các mẫu hoàn toàn tất định trông giống các hình huấn luyện hơn hay kém hơn so với phiên bản ngẫu nhiên?
3. Sinh có điều kiện theo lớp (không có điều kiện văn bản đầy đủ) đơn giản hơn để thêm vào: nối một vector lớp one-hot vào vector nhúng thời gian trước lớp `time_mlp`, huấn luyện trên dữ liệu dấu cộng/hình tròn có nhãn, và lấy mẫu riêng cho mỗi lớp. Điều kiện hóa theo lớp có làm thước đo đốm liên thông tốt hơn cho riêng hình dạng nào không?